# Streambased Core Demo — Hot / Cold Lifecycle

This walkthrough takes you through the lifecycle of data in Streambased, from kafka topic → seamless merged view → cold archive. Each step shows how the population shifts between tiers via a visualization.

In [1]:
import pandas as pd
import json
import subprocess
import time
import os
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [2]:
def q(sql):
    return spark.sql(sql).toPandas()

In [3]:
def compose(*args, capture=False):
    host_dir = os.environ["BREAKSTREAM_HOST_DIR"]
    cmd = [
        "docker", "compose",
        "--project-directory", f"{host_dir}/environment",
        "-f", f"{host_dir}/environment/docker-compose.yaml",
    ] + list(args)
    print(">", " ".join(cmd))
    result = subprocess.run(cmd, check=True, capture_output=capture, text=True)
    if capture:
        return result.stdout

In [4]:
def show_distribution(table='transactions'):
    hot_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM isk.hotset.{table}").iloc[0]
    cold_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM direct.coldset.{table}").iloc[0]
    merged_row = q(f"SELECT COUNT(*) AS cnt FROM isk.merged.{table}").iloc[0]

    def safe_int(v):
        if v is None or pd.isna(v):
            return None
        return int(v)

    hot_count = safe_int(hot_row['cnt']) or 0
    cold_count = safe_int(cold_row['cnt']) or 0
    merged_count = safe_int(merged_row['cnt']) or 0

    hot_earliest = safe_int(hot_row['earliest'])
    hot_latest = safe_int(hot_row['latest'])
    cold_earliest = safe_int(cold_row['earliest'])
    cold_latest = safe_int(cold_row['latest'])

    def fmt_offset(v):
        return f"{v:,}" if v is not None else '—'

    denom = max(merged_count, 1)
    hot_pct = round(100 * hot_count / denom)
    cold_pct = round(100 * cold_count / denom)

    total_pct = hot_pct + cold_pct
    if total_pct == 0:
        hot_bar_w = 0
        cold_bar_w = 0
        empty_bar = True
    else:
        hot_bar_w = hot_pct / total_pct * 100
        cold_bar_w = cold_pct / total_pct * 100
        empty_bar = False

    def bar_label(label, width):
        if width > 15:
            return f'<span style="color:#fff;font-size:12px;font-weight:600;pointer-events:none">{label}</span>'
        return ''

    if empty_bar:
        bar_html = '<div style="width:100%;height:32px;background:#e9ecef;border-radius:4px;"></div>'
    else:
        bar_html = f'''
        <div style="display:flex;width:100%;height:32px;border-radius:4px;overflow:hidden;">
          <div style="width:{hot_bar_w:.1f}%;background:#e63946;display:flex;align-items:center;justify-content:center;">
            {bar_label('Hot Set', hot_bar_w)}
          </div>
          <div style="width:{cold_bar_w:.1f}%;background:#4cc9f0;display:flex;align-items:center;justify-content:center;">
            {bar_label('Cold Set', cold_bar_w)}
          </div>
        </div>'''

    def svg_ring(pct, color):
        r = 40
        cx = cy = 50
        circumference = 2 * 3.14159 * r
        dash = circumference * min(pct, 100) / 100
        gap = circumference - dash
        return f'''
        <svg width="100" height="100" viewBox="0 0 100 100">
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="#e9ecef" stroke-width="8"/>
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="{color}" stroke-width="8"
            stroke-dasharray="{dash:.2f} {gap:.2f}"
            stroke-linecap="round"
            transform="rotate(-90 {cx} {cy})"/>
          <text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="central"
            style="font-size:16px;font-weight:700;fill:#222;">{pct}%</text>
        </svg>'''

    html = f'''
    <div style="font-family:system-ui,sans-serif;border:1.5px solid #ff6b35;border-radius:8px;padding:16px;max-width:680px;">
      <div style="margin-bottom:8px;">
        <span style="color:#ff6b35;font-weight:700;font-size:15px;">Merged Set</span>
        <span style="color:#222;font-size:15px;margin-left:16px;">Total Rows: <strong>{merged_count:,}</strong></span>
      </div>
      {bar_html}
      <div style="display:flex;gap:16px;margin-top:16px;">
        <div style="flex:1;border:1.5px solid #e63946;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#e63946;font-weight:700;font-size:14px;">Hot Set</span>
            <span style="color:#999;font-size:12px;">kafka</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(hot_pct, '#e63946')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Messages</td><td style="text-align:right;font-weight:600;">{hot_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(hot_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(hot_latest)}</td></tr>
          </table>
        </div>
        <div style="flex:1;border:1.5px solid #4cc9f0;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#4cc9f0;font-weight:700;font-size:14px;">Cold Set</span>
            <span style="color:#999;font-size:12px;">iceberg</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(cold_pct, '#4cc9f0')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Rows</td><td style="text-align:right;font-weight:600;">{cold_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(cold_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(cold_latest)}</td></tr>
          </table>
        </div>
      </div>
    </div>
    '''
    display(HTML(html))

## Initial state

Before we start traffic, here's what's already in place. The setup phase populated the coldset with seed data; no kafka traffic is flowing yet.

We begin by describing the databases available. We see:

* **hotset** — data from Kafka only
* **coldset** — data from Iceberg only
* **merged** — a hotset and coldset seamlessly unioned by Streambased.

Only the coldset database physically exists, hotset and merged are virtual databases provided by Streambased I.S.K.

In [5]:
q("SHOW DATABASES IN isk")

26/05/08 09:26:22 WARN AuthManagers: The property rest.sigv4-enabled is deprecated and will be removed in a future release. Please use the property rest.auth.type=sigv4 instead.


,namespace
0,hotset
1,coldset
2,merged
3,cdc_deletes


Let's list hotset tables. These correspond exactly to topics in Kafka.

If new topics are added they will immediately appear here.

The accounts table exists in Kafka only and has no Iceberg equivalent.

In [6]:
q("SHOW TABLES IN isk.hotset")

,namespace,tableName,isTemporary
0,hotset,accounts,False
1,hotset,customers,False
2,hotset,transactions,False


Now let's list coldset tables. These correspond exactly to tables in Iceberg.

Again, if new tables are added they will immediately appear here.

The branches table exists in Iceberg only and has no Kafka equivalent.

In [7]:
q("SHOW TABLES IN direct.coldset")

26/05/08 09:26:25 WARN AuthManagers: The property rest.sigv4-enabled is deprecated and will be removed in a future release. Please use the property rest.auth.type=sigv4 instead.


,namespace,tableName,isTemporary
0,coldset,branches,False


Finally, let's list merged tables. This is the union of topics in Kafka and tables in Iceberg.

Any that exist in both Kafka and Iceberg (e.g. transactions) are intelligently unioned by Streambased.

In [8]:
q("SHOW TABLES IN isk.merged")

,namespace,tableName,isTemporary
0,merged,accounts,False
1,merged,branches,False
2,merged,customers,False
3,merged,transactions,False


### Where does the data live right now?

In [9]:
show_distribution()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `direct`.`coldset`.`transactions` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 88;
'Aggregate [count(1) AS cnt#93L, 'MIN('kafka_offset) AS earliest#94, 'MAX('kafka_offset) AS latest#95]
+- 'UnresolvedRelation [direct, coldset, transactions], [], false


## Start the background traffic generator

We now produce continuous events into the kafka topic. Watch the hotset fill up while the coldset stays put.

In [15]:
compose('up', '-d', 'shadowtraffic_background')
time.sleep(8)

> docker compose --project-directory /home/roman/code/breakstream/environment -f /home/roman/code/breakstream/environment/docker-compose.yaml up -d shadowtraffic_background


 Container environment-zookeeper1-1  Running
 Container environment-kafka1-1  Recreate
 Container environment-kafka1-1  Recreated
 Container schema-registry  Recreate
 Container schema-registry  Recreated
 Container environment-shadowtraffic_background-1  Creating
 Container environment-shadowtraffic_background-1  Created
 Container environment-kafka1-1  Starting
 Container environment-kafka1-1  Started
 Container schema-registry  Starting
 Container schema-registry  Started
 Container schema-registry  Waiting
 Container schema-registry  Healthy
 Container environment-shadowtraffic_background-1  Starting
 Container environment-shadowtraffic_background-1  Started


To show the union, let's confirm the data size of the transactions topic on each view.

First the coldset, here we see the transactions data stored in Iceberg.

In [16]:
q("SELECT COUNT(*) AS coldset_count FROM direct.coldset.transactions")

,coldset_count
0,50000


Now the data size of the transactions topic in the hotset.

Here we see the transactions data stored in Kafka.

In [17]:
q("SELECT COUNT(*) AS hotset_count FROM isk.hotset.transactions")

,hotset_count
0,65


And now the data size of the transactions topic in the merged set.

This is a seamless combination of Kafka and Iceberg data.

Why doesn't the merged size equal hotset + coldset?

Streambased guarantees that any queries will run against the latest data available at the time of query execution. As the hotset is being constantly written to, more data is available in it now than there was when we queried the hotset individually.

In [18]:
q("SELECT COUNT(*) AS merged_count FROM isk.merged.transactions")

,merged_count
0,50072


### Distribution after traffic has been flowing for a few seconds

In [19]:
show_distribution()

Total Messages,65
Earliest Offset,"50,000"
Latest Offset,"50,064"
Total Rows,"50,000"
Earliest Offset,0
Latest Offset,"49,999"


## Pause traffic to examine schemas cleanly

Now let's look at schema evolution — for example addition of a new field. Note that background data injection is stopped now — to surface messages easier and inject data with evolved schema.

Let's describe the transactions table on the Hotset projection.

In [20]:
compose('down', 'shadowtraffic_background')
time.sleep(3)

> docker compose --project-directory /home/roman/code/breakstream/environment -f /home/roman/code/breakstream/environment/docker-compose.yaml down shadowtraffic_background


 Container environment-shadowtraffic_background-1  Stopping
 Container environment-shadowtraffic_background-1  Stopped
 Container environment-shadowtraffic_background-1  Removing
 Container environment-shadowtraffic_background-1  Removed
 Network environment_default  Removing
 Network environment_default  Resource is still in use


In [21]:
q("DESCRIBE isk.hotset.transactions")

,col_name,data_type,comment
0,TransactionID,string,None
1,AccountID,bigint,None
2,TransactionType,string,None
3,TransactionAmount,double,None
4,BranchID,bigint,None
5,CustomerFlaggedFraud,boolean,None
6,TransactionTime,timestamp_ntz,None
7,kafka_partition,int,None
8,kafka_offset,bigint,None
9,kafka_timestamp,timestamp_ntz,None


And fetch last few rows as an example.

Now — let's produce some new messages with additional field FraudRiskScore.

In [22]:
q("SELECT * FROM isk.hotset.transactions ORDER BY kafka_offset DESC LIMIT 10")

,TransactionID,AccountID,TransactionType,TransactionAmount,BranchID,CustomerFlaggedFraud,TransactionTime,kafka_partition,kafka_offset,kafka_timestamp
0,c53920a4-96d9-f06a-fdb9-e904204a67bc,1144,Withdrawal,8079.26,103,True,2025-10-23 12:35:07.312,0,50064,2026-05-08 09:20:09.950
1,5f03aed5-9841-9feb-5208-55b2d099a2df,1299,Deposit,9118.69,110,False,2025-10-23 12:35:06.979,0,50063,2026-05-08 09:20:09.617
2,04ed17fb-ba92-4ade-3a62-4c49e6ace728,1428,Deposit,1561.95,121,True,2025-10-23 12:35:06.646,0,50062,2026-05-08 09:20:09.283
3,7cddda5b-534f-74f1-26c5-4388f8b729ad,1267,Deposit,8055.15,121,True,2025-10-23 12:35:06.313,0,50061,2026-05-08 09:20:08.950
4,bcea6cc8-7e5e-de13-3c2a-079c5a3baf08,1180,Withdrawal,2245.16,121,False,2025-10-23 12:35:05.980,0,50060,2026-05-08 09:20:08.617
5,4e2b2383-9cc5-26ae-7c24-cf4828c7cb8c,1019,Deposit,8980.00,113,True,2025-10-23 12:35:05.647,0,50059,2026-05-08 09:20:08.283
6,399ac209-a640-4290-0db4-7bdd8d87850a,1042,Deposit,7940.68,115,True,2025-10-23 12:35:05.314,0,50058,2026-05-08 09:20:07.950
7,d2e1d607-82a4-31c2-dc1c-7739e42a4f41,1369,Deposit,712.85,103,True,2025-10-23 12:35:04.981,0,50057,2026-05-08 09:20:07.617
8,f2b90d54-6b6e-023c-57b9-79cfe2ca9959,1391,Withdrawal,8421.78,103,True,2025-10-23 12:35:04.648,0,50056,2026-05-08 09:20:07.283
9,faeaf9af-e8f0-5f55-b15d-e87d7ca4dfef,1432,Withdrawal,7077.28,121,True,2025-10-23 12:35:04.315,0,50055,2026-05-08 09:20:06.950


## Inject events with evolved schema

We now run a one-shot setup that adds a new column `FraudRiskScore` to the transactions topic, then start a new background generator that emits events with this evolved schema.

In [23]:
compose('up', '--wait', 'shadowtraffic_setup_evolved')

> docker compose --project-directory /home/roman/code/breakstream/environment -f /home/roman/code/breakstream/environment/docker-compose.yaml up --wait shadowtraffic_setup_evolved


 Container environment-zookeeper1-1  Running
 Container environment-kafka1-1  Running
 Container schema-registry  Running
 Container environment-shadowtraffic_setup_evolved-1  Recreate
 Container environment-shadowtraffic_setup_evolved-1  Recreated
 Container schema-registry  Waiting
 Container schema-registry  Healthy
 Container environment-shadowtraffic_setup_evolved-1  Starting
 Container environment-shadowtraffic_setup_evolved-1  Started
 Container environment-kafka1-1  Waiting
 Container schema-registry  Waiting
 Container environment-zookeeper1-1  Waiting
 Container environment-shadowtraffic_setup_evolved-1  Waiting
 Container environment-zookeeper1-1  Healthy
 Container environment-shadowtraffic_setup_evolved-1  Healthy
 Container environment-kafka1-1  Healthy
 Container schema-registry  Healthy


In [ ]:
compose('up', '-d', 'shadowtraffic_background_evolved')
time.sleep(8)

Now that new data is coming into the topic with evolved schema (FraudRiskScore optional field added) — let's describe the table again.

In [25]:
q("DESCRIBE isk.hotset.transactions")

,col_name,data_type,comment
0,TransactionID,string,None
1,AccountID,bigint,None
2,TransactionType,string,None
3,TransactionAmount,double,None
4,BranchID,bigint,None
5,CustomerFlaggedFraud,boolean,None
6,TransactionTime,timestamp_ntz,None
7,FraudRiskScore,double,None
8,kafka_partition,int,None
9,kafka_offset,bigint,None


And check the latest events added to it. Note the newest events have FraudRiskScore populated — while older ones are rendered with null (events pre-schema change).

Now we can restart ongoing data injection — but using evolved schema.

In [26]:
q("SELECT * FROM isk.hotset.transactions ORDER BY kafka_offset DESC LIMIT 15")

,TransactionID,AccountID,TransactionType,TransactionAmount,BranchID,CustomerFlaggedFraud,TransactionTime,FraudRiskScore,kafka_partition,kafka_offset,kafka_timestamp
0,4a9f8c55-326f-cb86-98d3-f1c22ff0e9c5,1103,Deposit,7357.29,121,True,2025-10-23 12:34:57.322,6.30,0,50144,2026-05-08 09:21:05.733
1,0b95bb0a-1a44-bf5e-c059-fd71c0d35d52,1369,Withdrawal,8674.69,121,True,2025-10-23 12:34:56.989,1.38,0,50143,2026-05-08 09:21:05.400
2,08e67c47-e492-cc54-3b2b-a5bd42231b49,1470,Deposit,8916.24,117,False,2025-10-23 12:34:56.656,3.41,0,50142,2026-05-08 09:21:05.066
3,6d9da8cd-0bb4-8fdc-09fd-f42d93e8e164,1326,Withdrawal,6282.27,103,False,2025-10-23 12:34:56.323,6.79,0,50141,2026-05-08 09:21:04.732
4,8285ef7f-6d0a-91f5-ced8-649a915870b2,1359,Withdrawal,8182.55,126,False,2025-10-23 12:34:55.990,1.41,0,50140,2026-05-08 09:21:04.399
5,7a1f9f71-bde0-1e0b-9a70-9b7121919a08,1097,Deposit,5638.12,126,False,2025-10-23 12:34:55.657,8.53,0,50139,2026-05-08 09:21:04.066
6,72269b9b-7f01-7da7-e83b-d1f3a15fd053,1398,Withdrawal,3356.88,102,True,2025-10-23 12:34:55.324,7.14,0,50138,2026-05-08 09:21:03.732
7,1e5e7741-c60f-670a-a418-84ee850c7d70,1388,Deposit,5993.47,123,True,2025-10-23 12:34:54.991,4.91,0,50137,2026-05-08 09:21:03.399
8,c226d0d1-c673-2c2f-ef5a-3eaf2fb8cd77,1457,Deposit,6086.54,106,True,2025-10-23 12:34:54.658,7.62,0,50136,2026-05-08 09:21:03.066
9,3eac5560-8587-0842-ad8a-3765d9435c1d,1106,Withdrawal,1797.43,125,True,2025-10-23 12:34:54.325,6.08,0,50135,2026-05-08 09:21:02.732


Right now our data is spread across Kafka and Iceberg. The Iceberg share (coldset) remains static in size but the Kafka share (hotset) is continuously growing.

Unbounded growth like this is often not optimal. Streambased allows you to use your Iceberg engine (in this case Spark) to easily transfer data from hotset to coldset.

Let's start by looking at the current data population in each set for the transactions table.

In [27]:
show_distribution()

Total Messages,145
Earliest Offset,"50,000"
Latest Offset,"50,144"
Total Rows,"50,000"
Earliest Offset,0
Latest Offset,"49,999"


Moving data from Kafka to Iceberg is a simple `INSERT INTO ... SELECT FROM` statement with Streambased — we simply select from the hotset into the coldset.

Transferring data in this way has the following advantages:

* Transfer from hotset to coldset is not a pre-requisite for data access
* It can be scheduled during low resource usage periods.
* Data can be accumulated in Kafka until it is efficient to transfer it (no small files).
* The transfer process is atomic and involves no downtime.

But first we need to add the new column to the coldset — reflecting the Kafka schema evolution.

In [28]:
q("ALTER TABLE direct.coldset.transactions ADD COLUMN FraudRiskScore double AFTER TransactionTime")

""


In [29]:
q("DESCRIBE direct.coldset.transactions")

,col_name,data_type,comment
0,TransactionID,string,None
1,AccountID,bigint,None
2,TransactionType,string,None
3,TransactionAmount,double,None
4,BranchID,bigint,None
5,CustomerFlaggedFraud,boolean,None
6,TransactionTime,timestamp_ntz,None
7,FraudRiskScore,double,None
8,kafka_partition,int,None
9,kafka_offset,bigint,None


And finally move the data using the `INSERT INTO ... SELECT FROM` statement.

In [ ]:
q("INSERT INTO direct.coldset.transactions SELECT * FROM isk.hotset.transactions")

After transfer the coldset population has increased. Streambased will now serve a larger share of the merged dataset from the coldset.

Why hasn't the hotset population decreased?

Streambased will not explicitly delete data from Kafka. Kafka deletion is handled by the topic retention policy as usual.

The merged set now contains all of the coldset + the section of hotset that has not yet been transferred to the coldset, as you can see below.

In [ ]:
show_distribution()

The hot/cold lifecycle is complete. Data originated in Kafka (hotset), was seamlessly unified with existing Iceberg data in the merged view, and has now been durably archived to the coldset — all without any downtime or schema lock-in.

From here you can explore further: run time-travel queries against the coldset snapshots, inspect Kafka lag independently of query latency, or try the advanced demos covering KSI-accelerated lookups and Slipstream change feeds.